In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != "ic" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / "data"


In [1]:
import pandas as pd
import re
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import statsmodels.api as sm
import numpy as np


In [2]:
df = pd.read_parquet(DATA_DIR / 'tweets_192k_labeled.parquet')


In [3]:
df.head()

,full_text,clean_text,unsupervised_sentiment
0,se o lula ganhar eu quero uma roda de beijo co...,se o lula ganhar eu quero uma roda de beijo co...,1.0
1,@ixslorena LULA PRESIDENTE HOJE,@USER LULA PRESIDENTE HOJE,0.0
2,mãe morrendo de alegria na carreata do Lula,mãe morrendo de alegria na carreata do Lula,0.0
3,@trajaza AVISA QUE MEU PAI LULA VAI GANHAR PRI...,@USER AVISA QUE MEU PAI LULA VAI GANHAR PRIMEI...,0.0
4,@indieoffw O RJ elegendo castro e juram que va...,@USER O RJ elegendo castro e juram que vai dar...,0.0


In [4]:
# Segui com o df_model

df_model = df.copy()
df_model["search_text"] = (
    df_model["full_text"].fillna("").astype(str) + " " +
    df_model["clean_text"].fillna("").astype(str)
).str.lower()

In [12]:
len(df_model)

192954

In [ ]:

candidate_patterns = {
    "lula": r"\blula\b|\blule\b|\blulinha\b|\blulão\b|\bpt\b|\b9 dedos\b|\bnove dedos\b|\b13\b",

    "bolsonaro": r"\bbolsonaro\b|\bbolsonario\b|\bbolsonaru\b|\bbozo\b|\bbozonaro\b|\bbolsolixo\b|\bbolsominion\b|\bjair\b|\bbiroliro\b|\bmito\b",

    "tebet": r"\btebet\b|\bsimone tebet\b|\bsimone\b",

    "ciro_gomes": r"\bciro\b|\bciro gomes\b|\bcirogomes\b|\bciro2022\b",

    
    }
   

In [6]:
economic_words = [
        'emprego','desemprego','bolsa','bolsa de valores','mercado','mercado financeiro',
        'inflacao','inflação','renda','dolar','dólar','cambio','câmbio','taxa de cambio','taxa de câmbio',
        'juros','taxa de juros','industria','indústria','salario','salarios','salários','valor',
        'auxilio','auxílio','banco','bancos','credito','crédito','desigual','desigualdade',
        'empreendedor','empreendedorismo','financeiro','financas','finanças','financa',
        'financiamento','financiar','impostos','imposto','economia','igualdade',
        'pib','produto interno bruto','recessao','recessão','crescimento','deflacao','deflação',
        'superavit','superávit','deficit','déficit','orcamento','orçamento','orcamento publico','orçamento público',
        'gasto publico','gasto público','divida','dívida','divida publica','dívida pública',
        'investimento','investimentos','investidor','investidores','acoes','ações','acao','ação',
        'derivativos','derivativo','futuros',
        'b3','selic','copom','tesouro','tesouro direto','banco central',
        'custo de vida','pobreza','renda minima','renda mínima','salario minimo','salário mínimo',
        'mercado de trabalho','carteira assinada','clt','educacao','educação'
]

In [7]:

for candidate, pattern in candidate_patterns.items():
    df_model[f"mentions_{candidate}"] = (
        df_model["search_text"].str.contains(pattern, regex=True, na=False).astype(int)
    )

In [8]:
mention_cols = [f"mentions_{candidate}" for candidate in candidate_patterns.keys()]
df_candidates = df_model[df_model[mention_cols].sum(axis=1) > 0].copy()


In [9]:
df_candidates["mentioned_candidates"] = df_candidates[mention_cols].apply(
    lambda row: [
        col.replace("mentions_", "")
        for col, value in row.items()
        if value == 1
    ],
    axis=1
) 

In [10]:
df_candidates = df_candidates[
    ["full_text", "clean_text", "unsupervised_sentiment", "mentioned_candidates"] + mention_cols
].copy()

In [11]:
len(df_candidates)

190300

In [11]:
df_candidates.head()

,full_text,clean_text,unsupervised_sentiment,mentioned_candidates,mentions_lula,mentions_bolsonaro,mentions_tebet,mentions_ciro_gomes
0,se o lula ganhar eu quero uma roda de beijo co...,se o lula ganhar eu quero uma roda de beijo co...,1.0,[lula],1,0,0,0
1,@ixslorena LULA PRESIDENTE HOJE,@USER LULA PRESIDENTE HOJE,0.0,[lula],1,0,0,0
2,mãe morrendo de alegria na carreata do Lula,mãe morrendo de alegria na carreata do Lula,0.0,[lula],1,0,0,0
3,@trajaza AVISA QUE MEU PAI LULA VAI GANHAR PRI...,@USER AVISA QUE MEU PAI LULA VAI GANHAR PRIMEI...,0.0,[lula],1,0,0,0
4,@indieoffw O RJ elegendo castro e juram que va...,@USER O RJ elegendo castro e juram que vai dar...,0.0,[lula],1,0,0,0


In [12]:

mention_cols = [
    "mentions_lula",
    "mentions_bolsonaro",
    "mentions_tebet",
    "mentions_ciro_gomes",
    
]

In [13]:
candidate_mention_counts = (
    df_candidates[mention_cols]
    .sum()
    .sort_values(ascending=False)
    .rename("mention_count")
    .reset_index()
)

In [14]:
candidate_mention_counts["candidate"] = candidate_mention_counts["index"].str.replace("mentions_", "", regex=False)
candidate_mention_counts = candidate_mention_counts[["candidate", "mention_count"]]



In [15]:
sentiment_by_candidate = {}

In [16]:
for col in mention_cols:
    candidate_name = col.replace("mentions_", "")
    temp = (
        df_candidates[df_candidates[col] == 1]["unsupervised_sentiment"]
        .value_counts(dropna=False)
        .rename_axis("unsupervised_sentiment")
        .reset_index(name="count")
    )
    temp["candidate"] = candidate_name
    sentiment_by_candidate[candidate_name] = temp[["candidate", "unsupervised_sentiment", "count"]]


In [17]:
sentiment_candidate_table = pd.concat(sentiment_by_candidate.values(), ignore_index=True)

In [18]:
sentiment_candidate_pivot = (
    sentiment_candidate_table
    .pivot(index="candidate", columns="unsupervised_sentiment", values="count")
    .fillna(0)
    .astype(int)
)

In [19]:
sentiment_candidate_pivot["total_mentions"] = sentiment_candidate_pivot.sum(axis=1)
sentiment_candidate_pivot = sentiment_candidate_pivot.sort_values("total_mentions", ascending=False)

In [20]:
print(candidate_mention_counts)
print(sentiment_candidate_pivot)


    candidate  mention_count
0        lula         118974
1   bolsonaro          75646
2  ciro_gomes          19898
3       tebet           4139
unsupervised_sentiment   -1.0    0.0    1.0  total_mentions
candidate                                                  
lula                    37412  34972  46590          118974
bolsonaro               34439  21051  20156           75646
ciro_gomes               8526   3342   8030           19898
tebet                    1527   1383   1229            4139


In [21]:
pattern = r"\b(?:{})\b".format(
    "|".join(re.escape(term.lower()) for term in economic_words)
)

df_candidates["is_economic"] = (
    df_candidates["clean_text"]
    .fillna("")
    .str.lower()
    .str.contains(pattern, regex=True)
    .astype(int)
)

In [22]:
df_candidates["is_economic"].value_counts()


is_economic
0    186656
1      3644
Name: count, dtype: int64

In [23]:
df_candidates.head()

,full_text,clean_text,unsupervised_sentiment,mentioned_candidates,mentions_lula,mentions_bolsonaro,mentions_tebet,mentions_ciro_gomes,is_economic
0,se o lula ganhar eu quero uma roda de beijo co...,se o lula ganhar eu quero uma roda de beijo co...,1.0,[lula],1,0,0,0,0
1,@ixslorena LULA PRESIDENTE HOJE,@USER LULA PRESIDENTE HOJE,0.0,[lula],1,0,0,0,0
2,mãe morrendo de alegria na carreata do Lula,mãe morrendo de alegria na carreata do Lula,0.0,[lula],1,0,0,0,0
3,@trajaza AVISA QUE MEU PAI LULA VAI GANHAR PRI...,@USER AVISA QUE MEU PAI LULA VAI GANHAR PRIMEI...,0.0,[lula],1,0,0,0,0
4,@indieoffw O RJ elegendo castro e juram que va...,@USER O RJ elegendo castro e juram que vai dar...,0.0,[lula],1,0,0,0,0


In [26]:
df_candidates.to_parquet(DATA_DIR / 'binary_model_table.parquet')

In [24]:
X = df_candidates[mention_cols + ["is_economic"]].copy()
y = df_candidates["unsupervised_sentiment"].copy()


In [25]:
# Check exact sentiment labels before running
print("Unique sentiment labels:", y.unique())

Unique sentiment labels: [ 1.  0. -1.]


In [28]:
X_sm = sm.add_constant(X)

# Modelo logit para sentimento positivo (y) binaria; x binaria p/ presidentes + is_economic
logit_pos = sm.Logit((y == 1).astype(int), X_sm).fit(cov_type="HC3")

result_positive = logit_pos

print("\nPOSITIVE MODEL")
print(result_positive.summary())


Optimization terminated successfully.
         Current function value: 0.640094
         Iterations 5

POSITIVE MODEL
                             Logit Regression Results                             
Dep. Variable:     unsupervised_sentiment   No. Observations:               190300
Model:                              Logit   Df Residuals:                   190294
Method:                               MLE   Df Model:                            5
Date:                    Wed, 15 Apr 2026   Pseudo R-squ.:                 0.02257
Time:                            18:05:46   Log-Likelihood:            -1.2181e+05
converged:                           True   LL-Null:                   -1.2462e+05
Covariance Type:                      HC3   LLR p-value:                     0.000
                          coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------
const                  -0.0312      0.017 

In [85]:
print("\nOdds Ratios - Positive")
print(np.exp(result_positive.params))



Odds Ratios - Positive
const                  0.969260
mentions_lula          0.758501
mentions_bolsonaro     0.404746
mentions_tebet         0.668540
mentions_ciro_gomes    0.934253
is_economic            0.756644
dtype: float64


In [ ]:
# Modelo logit para sentimento neutro (y) binaria; x binaria p/ presidentes + is_economic

logit_neu = sm.Logit((y == 0).astype(int), X_sm).fit()
result_neutral = logit_neu
print("Neutral Model: ")
print(result_neutral.summary())


Optimization terminated successfully.
         Current function value: 0.584575
         Iterations 6
Neutral Model: 
                             Logit Regression Results                             
Dep. Variable:     unsupervised_sentiment   No. Observations:               190300
Model:                              Logit   Df Residuals:                   190294
Method:                               MLE   Df Model:                            5
Date:                    Wed, 01 Apr 2026   Pseudo R-squ.:                 0.01132
Time:                            17:24:06   Log-Likelihood:            -1.1124e+05
converged:                           True   LL-Null:                   -1.1252e+05
Covariance Type:                nonrobust   LLR p-value:                     0.000
                          coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------
const                  -0.9952      0.017 

In [89]:
print("\nOdds Ratios - Neutral")
print(np.exp(result_neutral.params))



Odds Ratios - Neutral
const                  0.369659
mentions_lula          1.151276
mentions_bolsonaro     1.046893
mentions_tebet         2.367470
mentions_ciro_gomes    0.441902
is_economic            0.396131
dtype: float64


In [ ]:
# Modelo logit para sentimento negativo (y) binaria; x binaria p/ presidentes + is_economic

logit_neg = sm.Logit((y == -1).astype(int), X_sm).fit()
result_negative = logit_neg
print("Negative Model: ")
print(result_negative.summary())



Optimization terminated successfully.
         Current function value: 0.635151
         Iterations 5
Negative Model: 
                             Logit Regression Results                             
Dep. Variable:     unsupervised_sentiment   No. Observations:               190300
Model:                              Logit   Df Residuals:                   190294
Method:                               MLE   Df Model:                            5
Date:                    Wed, 01 Apr 2026   Pseudo R-squ.:                 0.02729
Time:                            17:25:11   Log-Likelihood:            -1.2087e+05
converged:                           True   LL-Null:                   -1.2426e+05
Covariance Type:                nonrobust   LLR p-value:                     0.000
                          coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------
const                  -1.0636      0.016

In [91]:
print("\nOdds Ratios - Negative")
print(np.exp(result_negative.params))


Odds Ratios - Negative
const                  0.345212
mentions_lula          1.115330
mentions_bolsonaro     2.255197
mentions_tebet         0.727097
mentions_ciro_gomes    1.849779
is_economic            2.360620
dtype: float64
